# Selección y parametrización de distribuciones usando datos del mundo real

> **Nota**  
> ¡Muchas gracias a [Richard Pilbery](https://github.com/RichardPilbery) por señalarme este excelente paquete!

Más allá de usar algunas distribuciones estándar que suelen ajustarse bien a ciertos fenómenos, como por ejemplo:

- **lognormal** para tiempos de actividad
- **exponencial** para tiempos entre llegadas

y de parametrizarlas con los promedios (y, cuando aplique, medidas de dispersión) obtenidos de nuestros datos reales, podemos ir un paso más allá y emplear una distribución con una **forma diferente** que refleje aún más fielmente nuestros datos del mundo real.

## El paquete *fitter*

En esta sección utilizaremos el paquete [**fitter**](https://github.com/cokelaer/fitter).

Si aún no lo tienes instalado, ejecuta:

```
pip install fitter
```

*fitter* proporciona funciones de ayuda para encontrar la **distribución más apropiada** para tus datos reales y, después, devolver la distribución y todos los **parámetros** relevantes.

### Una demostración rápida de *fitter*

Comencemos con un CSV de ejemplo de tiempos históricos de actividad.

```{python}
#| eval: false
#| echo: false

import pandas as pd

df = pd.read_csv("resources/complex_event_log.csv")

df = df[df["event"].isin(["MINORS_examination_begins", "MINORS_examination_complete"])]

df = df[["entity_id","run", "event", "time"]]

df = df.pivot(index=["entity_id", "run"], columns="event", values="time").reset_index()

df["duration"] = df["MINORS_examination_complete"] - df["MINORS_examination_begins"]

df["P_ID"] = df.apply(lambda x: f'{x["entity_id"]:.0f}_{x["run"]:.0f}', axis=1)

df = df[~df["duration"].isna()]

df[["P_ID","duration"]].to_csv("resources/MINORS_examination_duration.csv", index=False)
```

Aquí tenemos un *dataframe* con la duración histórica de un paso concreto de un proceso: la duración de una cita con enfermería.

```{python}
import pandas as pd

df = pd.read_csv("resources/MINORS_examination_duration.csv")

df.head()
```

Para empezar, **visualicémosla**.

```{python}
import plotly.express as px

px.histogram(df["duration"].round(1))
```

A primera vista, parece una **normal**: una curva acampanada con colas aproximadamente simétricas. Pero ¿qué parámetros necesitamos? ¿Y es *seguro* que sea normal?

> **Consejo**  
> Reflexiona cuidadosamente sobre las variables para las que decides ajustar distribuciones.
>
> Probablemente quieras muestrear aleatoriamente distribuciones para **tus duraciones de actividad** y **tus tiempos entre llegadas**; pero **no** querrás usar distribuciones históricas de **tiempos de espera**, ya que estos deberían **emerger** del diseño de tu simulación (número de recursos, probabilidades de que las personas realicen distintas actividades, etc.) y no venir impuestos por datos históricos, que además suelen reflejar **actividad** y no **demanda** pura.

### Limitar *fitter* a distribuciones comunes

Por defecto, *fitter* examina **todas** las distribuciones que ofrece la librería *scipy*.

Esto puede hacer que te recomiende distribuciones **poco habituales** —y quizá menos apropiadas—.

En su lugar, es recomendable limitar *fitter* a un **conjunto básico** de distribuciones.

```{python}
distributions_to_scan = [
            "poisson",
            "bernoulli",
            "triang",
            "erlang",
            "weibull_min",
            "expon_weib",
            "betabinom",
            "pearson3",
            "cauchy",
            "chi2",
            "expon",
            "exponpow",
            "gamma",
            "lognorm",
            "norm",
            "powerlaw",
            "rayleigh",
            "uniform"
        ]
```

> **Nota**  
> Más adelante hablamos del paquete `sim-tools` como alternativa a *scipy* para gestionar distribuciones.
>
> En ese caso, te convendría pasar la lista de distribuciones admitidas por **sim-tools** en su lugar.
>
> Puedes encontrar dicha lista [aquí](https://tommonks.github.io/sim-tools/01_sampling/01_distributions_examples.html#summary-of-implemented-distributions).

## Ajuste con *fitter*

```{python}
from fitter import Fitter

f = Fitter(df["duration"], distributions=distributions_to_scan, timeout=60)

f.fit()
```

El método `summary` devuelve una salida con varias distribuciones candidatas y **qué tan bien** se ajustan.

En este ejemplo, **gamma**, **erlang**, **lognorm**, **pearson3** y **norm** parecen aproximar muy bien los datos; la **gamma** resulta ser el **mejor ajuste**, aunque la diferencia es pequeña.

```{python}
nurse_appt_duration_fit = f.summary()

nurse_appt_duration_fit
```

> **Consejo**  
> Cada columna mide la **calidad del ajuste**; algunas penalizan además la **complejidad** excesiva.
>
> Mejor cuanto **más pequeño**: `sumsquare_error`, `aic`, `bic`, `ks_statistic`  
> Mejor cuanto **más grande**: `ks_pvalue`

Ahora extraigamos los detalles de la **mejor** distribución.

Esto devuelve un **diccionario** con los parámetros necesarios para configurar la distribución con alguna de las librerías de Python para distribuciones.

```{python}
nurse_appt_duration_fit = f.get_best()

nurse_appt_duration_fit
```

## Definir una distribución

Con esos parámetros podemos configurar y muestrear una distribución **gamma**.

### *scipy*

La librería `random` ¡no tiene distribución gamma!

Además, `random` presenta limitaciones para lograr **reproducibilidad** y **reducción de varianza** entre corridas al comparar escenarios (temas que trataremos en un capítulo posterior sobre reproducibilidad).

Por estas razones, en lugar de `random` usaremos las distribuciones de **scipy**.

```{python}
from scipy.stats import gamma

# Ejemplo: generar una muestra aleatoria
sample = gamma.rvs(
  a=nurse_appt_duration_fit['gamma']["a"],
  loc=nurse_appt_duration_fit['gamma']["loc"],
  scale=nurse_appt_duration_fit['gamma']["scale"]
  )

sample
```

De forma alternativa, podemos escribir algo así:

```{python}
def gamma_duration():
    return gamma.rvs(
        a=nurse_appt_duration_fit['gamma']["a"],
        loc=nurse_appt_duration_fit['gamma']["loc"],
        scale=nurse_appt_duration_fit['gamma']["scale"]
        )
```

Luego, cada vez que necesitemos una duración, simplemente ejecutamos:

```{python}
gamma_duration()
```

### *sim-tools*

También puedes considerar el paquete **sim-tools**, que ofrece utilidades adicionales para gestionar **distribuciones** y **aleatoriedad**.

Puedes saber más sobre *sim-tools* [aquí](https://tommonks.github.io/sim-tools/00_front_page.html).

Para usar *sim-tools*, primero ejecuta:

```
pip install sim-tools
```

> **Advertencia**  
> Observa que el nombre del paquete lleva **guion** (`sim-tools`), pero al importarlo se usa **guion bajo**: `sim_tools`.

```{python}
from sim_tools.distributions import Gamma

# Definir una distribución
my_gamma = Gamma(
  alpha=nurse_appt_duration_fit['gamma']["a"],
  location=nurse_appt_duration_fit['gamma']["loc"],
  beta=nurse_appt_duration_fit['gamma']["scale"], # En sim-tools el parámetro 'scale' se llama 'beta'
  random_seed=42
  )

# Obtener una muestra aleatoria
sample = my_gamma.sample()

sample
```

> **Consejo**  
> En una simulación completa, puede ser útil configurar la distribución de **cada tiempo de actividad**, **tiempo entre llegadas** y otros **puntos de decisión** en el método `__init__` de tu clase `Model`.
>
> Cada vez que necesites muestrear, basta con llamar al método `.sample()` y, gracias a la semilla aleatoria, obtendrás resultados **reproducibles**.
>
> Esa es la idea general; lo cubrimos en detalle en el capítulo de **reproducibilidad** (*reproducibility.qmd*), o puedes revisar la clase [**DistributionRegistry**](https://tommonks.github.io/sim-tools/01_sampling/03_distributions_registry.html) en *sim-tools* para una gestión aún más robusta.

## Una prueba rápida de que funciona

¡Demostremos que los tres métodos producen algo similar!

```{python}
# Generate samples from each method
# We'll use len(df["duration"]) to generate as many samples as we have in our real (historical) dataset
samples_scipy = [gamma_duration() for _ in range(len(df["duration"]))]
samples_simtools = [my_gamma.sample() for _ in range(len(df["duration"]))]
```

```{python}
#| code-fold: true
#| code-summary: "Show the code"
#|
import plotly.graph_objects as go

fig = go.Figure()

# Real data histogram
fig.add_trace(go.Histogram(
    x=df["duration"], # Our 'real' data
    nbinsx=50,
    name='Historical Data',
    opacity=0.3,
    marker_color='green'
))

# Scipy samples histogram
fig.add_trace(go.Histogram(
    x=samples_scipy,
    nbinsx=50,
    name='scipy.stats.gamma Samples',
    opacity=0.3,
    marker_color='blue'
))

# sim_tools samples histogram
fig.add_trace(go.Histogram(
    x=samples_simtools,
    nbinsx=50,
    name='sim_tools Gamma Samples',
    opacity=0.3,
    marker_color='red'
))

# Overlay histograms
fig.update_layout(
    barmode='overlay',
    title_text='Comparison of Sample Distributions',
    xaxis_title_text='Duration',
    yaxis_title_text='Count',
    legend_title_text='Data Source'
)

fig.show()
```

```{python}
#| code-fold: true
#| code-summary: "Show the code"

from scipy.stats import gaussian_kde
import numpy as np

# --- Create density estimations ---
# Use scipy's gaussian_kde for smooth density curves
real_data = df["duration"].copy()

kde_real = gaussian_kde(real_data)
kde_scipy = gaussian_kde(samples_scipy)
kde_simtools = gaussian_kde(samples_simtools)

# Define a common x-axis range (based on min/max of all data)
xmin = min(real_data.min(), min(samples_scipy), min(samples_simtools))
xmax = max(real_data.max(), max(samples_scipy), max(samples_simtools))
x_values = np.linspace(xmin, xmax, 500)

# --- Create plot ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x_values,
    y=kde_real(x_values),
    mode='lines',
    name='Historical Data',
    line=dict(color='green')
))

fig.add_trace(go.Scatter(
    x=x_values,
    y=kde_scipy(x_values),
    mode='lines',
    name='scipy.stats.gamma Samples',
    line=dict(color='blue')
))

fig.add_trace(go.Scatter(
    x=x_values,
    y=kde_simtools(x_values),
    mode='lines',
    name='sim_tools Gamma Samples',
    line=dict(color='red')
))

# Layout
fig.update_layout(
    title='Density Plot Comparison of Real and Simulated Data',
    xaxis_title='Duration',
    yaxis_title='Density',
    legend_title_text='Source'
)

fig.show()

```

## Resumen

Hemos visto algunas librerías que ayudan a garantizar que las distribuciones de tu simulación **reflejen con precisión** los datos del mundo real.

Ahora podrías repetir este proceso para **cada tiempo de actividad** de tus datos, así como para los **tiempos entre llegadas**, y comprobar cuánto cambia con respecto a usar las distribuciones **exponencial** y **lognormal** que hemos utilizado hasta ahora en el libro.
